# Multi-Touch Attribution Modeling

**Project:** Marketing Attribution & ROI Dashboard (Infotact Internship)

**Goal:** Engineer touchpoint-level features from campaign/ad interaction data and apply three attribution models — First-Touch, Last-Touch, and Linear — to distribute conversion credit across marketing channels.

**Inputs:** Cleaned marketing campaign performance dataset (10K rows) with user/session-level touchpoints.

**Outputs:** `attribution_results.csv`, `channel_attribution_summary.csv` — feeds directly into the Power BI dashboard.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_columns', None)
sns.set_style('whitegrid')

## 1. Load Data

Replace `DATA_PATH` with the path to your cleaned campaign performance CSV (the one used in the SQL/EDA stage).

In [ ]:
DATA_PATH = '../data/marketing_campaign_performance_clean.csv'

df = pd.read_csv(DATA_PATH, parse_dates=['touchpoint_date'])

# Expected columns (adjust to match your actual schema):
# user_id, touchpoint_date, channel, campaign_id, converted (0/1), conversion_date

df.head()

## 2. Feature Engineering: Touchpoint Sequencing

These features are specific to attribution modeling (separate from the EDA-stage features like CTR, CAC, Conversion Rate already engineered earlier):

- `touch_order` — sequence number of each touchpoint per user
- `path_length` — total number of touchpoints in a user's journey
- `is_first_touch` / `is_last_touch` — boolean flags marking journey boundaries
- `days_since_first_touch` — recency feature relative to journey start

In [ ]:
df = df.sort_values(['user_id', 'touchpoint_date']).reset_index(drop=True)

# Sequence number of each touchpoint within a user's journey
df['touch_order'] = df.groupby('user_id').cumcount() + 1

# Total path length per user
path_length = df.groupby('user_id')['touch_order'].transform('max')
df['path_length'] = path_length

# First / last touch flags
df['is_first_touch'] = df['touch_order'] == 1
df['is_last_touch'] = df['touch_order'] == df['path_length']

# Days since first touch (recency within journey)
first_touch_date = df.groupby('user_id')['touchpoint_date'].transform('min')
df['days_since_first_touch'] = (df['touchpoint_date'] - first_touch_date).dt.days

df.head(10)

In [ ]:
print('Total users:', df['user_id'].nunique())
print('Total touchpoints:', len(df))
print('Avg path length:', round(df.groupby('user_id')['path_length'].first().mean(), 2))
print('Converted users:', df.loc[df['converted'] == 1, 'user_id'].nunique())

## 3. Filter to Converting Paths

Attribution credit is only distributed across journeys that ended in a conversion.

In [ ]:
converted_users = df.loc[df['converted'] == 1, 'user_id'].unique()
paths = df[df['user_id'].isin(converted_users)].copy()

print(f'Converting journeys: {paths["user_id"].nunique()}')
print(f'Touchpoints in converting journeys: {len(paths)}')

## 4. Attribution Models

Each model assigns a `credit` value (summing to 1.0 per converting user) to every touchpoint in that user's journey.

In [ ]:
def first_touch_attribution(paths):
    out = paths.copy()
    out['credit'] = 0.0
    out.loc[out['is_first_touch'], 'credit'] = 1.0
    out['model'] = 'first_touch'
    return out

def last_touch_attribution(paths):
    out = paths.copy()
    out['credit'] = 0.0
    out.loc[out['is_last_touch'], 'credit'] = 1.0
    out['model'] = 'last_touch'
    return out

def linear_attribution(paths):
    out = paths.copy()
    out['credit'] = 1.0 / out['path_length']
    out['model'] = 'linear'
    return out

In [ ]:
ft = first_touch_attribution(paths)
lt = last_touch_attribution(paths)
lin = linear_attribution(paths)

attribution_results = pd.concat([ft, lt, lin], ignore_index=True)

# Sanity check: credit per user per model should sum to 1.0
check = attribution_results.groupby(['user_id', 'model'])['credit'].sum().reset_index()
assert np.allclose(check['credit'], 1.0), 'Credit does not sum to 1.0 for some user/model combos'
print('Credit sums validated — all converting journeys total 1.0 per model.')

attribution_results.head(10)

## 5. Channel-Level Attribution Summary

Aggregate credit by channel and model — this is the table the Power BI dashboard will visualize.

In [ ]:
channel_summary = (
    attribution_results
    .groupby(['model', 'channel'])['credit']
    .sum()
    .reset_index()
    .rename(columns={'credit': 'attributed_conversions'})
    .sort_values(['model', 'attributed_conversions'], ascending=[True, False])
)

channel_summary

## 6. Visualization: Channel Credit by Model

In [ ]:
pivot = channel_summary.pivot(index='channel', columns='model', values='attributed_conversions').fillna(0)
pivot = pivot[['first_touch', 'last_touch', 'linear']]

ax = pivot.plot(kind='bar', figsize=(10, 6))
plt.title('Attributed Conversions by Channel and Model')
plt.ylabel('Attributed Conversions')
plt.xlabel('Channel')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Model')
plt.tight_layout()
plt.savefig('channel_attribution_comparison.png', dpi=150)
plt.show()

## 7. Export for Power BI

In [ ]:
attribution_results.to_csv('attribution_results.csv', index=False)
channel_summary.to_csv('channel_attribution_summary.csv', index=False)

print('Exported: attribution_results.csv, channel_attribution_summary.csv')

## Notes for the team

- `attribution_results.csv` is row-level (one row per touchpoint per model) — use it if the dashboard needs path-level drill-down.
- `channel_attribution_summary.csv` is pre-aggregated by channel and model — use this as the primary Power BI data source for the channel comparison visuals, it's lighter and faster to load.
- `credit` always sums to 1.0 per converting user per model — this lets you safely sum credit across users to get total attributed conversions per channel.
- Next step: join `channel_attribution_summary.csv` with ad spend data to compute attributed ROI per channel per model.